In [ ]:
import matplotlib.pyplot as plt
import matplotlib as mpl
from itertools import product
import numpy as np
import seaborn as sns

#Now we will create a markov heat map to inspect connections when we
#change from one chord (roman number) to the other.
#Naturally and mathematically due to the limitations/laws of baroque harmony,
#the heatmap will display certain strong connections such as vii to I.

#First of all we will need a list for each chorale with the roman chord numbers
#in order to get the chord transactions for every song
transaction_matrix = []
for i, j in roman_chord_analysis_df.groupby("chorale_name"):
  transaction_matrix.append(j["roman_scale_degree"].tolist())

transactions = []
for i in transaction_matrix:
  for j in zip(i[:-1], i[1:]):
    transactions.append(j)

#the x and y axis for the Markov-Transaction heatmap
index = roman_chord_analysis_df["roman_scale_degree"].unique()
columns = index

df = pd.DataFrame(np.zeros(shape=(len(columns), len(index))),
                  columns=columns, index=index, dtype=int)

for idx,col in product(index, columns):
    df[col].loc[idx] = transactions.count((idx, col))

In [ ]:
import numpy as np, matplotlib.pyplot as plt, seaborn as sns
from matplotlib.colors import PowerNorm

#Ordering the dataset for a more easy-to-read result
order = ["I", "II", "III", "IV", "V", "VI", "VII", "i", "ii",
         "iii", "iv", "v", "vi", "vii"]
Order = []
counts = df.copy()

for i in order:
  if i in counts.index:
    Order.append(i)

counts = counts.loc[Order, Order]

#Creating the heatmap with the number of occurrences of each chord transaction
#Width and height of the heatmap
plt.figure(figsize=(14, 8))
#seting dataset, color and annotation
sns.heatmap(counts, cmap="BuPu", annot=True, fmt="d")
plt.title("Heat Map of the Changes Between Chords")#title
plt.ylabel("First Chord")
plt.xlabel("Second Chord")
plt.yticks(rotation=0)#No rotation for the yaxis chord titles
plt.show()

#The same heatmap, but with the probability that the second chord will occur,
#given that the first chord has occured
probs = counts.div(counts.sum(axis=1), axis=0).fillna(0)

fig, ax = plt.subplots(figsize=(16, 7.5))
hm = sns.heatmap(probs, cmap="BuPu", annot=True, fmt=".2f",
                 annot_kws={"size": 18}, vmin=0, ax=ax,
                 cbar_kws={"label": "P(second | first)"})

cbar = hm.collections[0].colorbar
cbar.ax.tick_params(labelsize=20)          # colorbar tick numbers

ax.set_title("Chord Transition Probabilities",
             fontsize=20, pad=15)
ax.set_ylabel("First Chord",  fontsize=20)
ax.set_xlabel("Second Chord", fontsize=20)
ax.tick_params(labelsize=14)
ax.collections[0].colorbar.set_label("P(second | first)", size=20)
plt.yticks(rotation=0)
plt.tight_layout()
plt.savefig("hmap.png", dpi=300, bbox_inches="tight", transparent=True)
plt.show()                                         # savefig must always come BEFORE show

#Cross Entropy using the Markov Matrix with 2 levels
#Laplace smoothing, otherwise we will have log(0) = -inf
smoothed = (counts + 1).div((counts + 1).sum(axis=1), axis=0)
logs = [np.log(smoothed.loc[i, j]) for i, j in transactions]
m_cross_entropy = -np.mean(logs)
print(f"CE = {m_cross_entropy:.3f}  PP = {np.exp(m_cross_entropy):.2f}")

A cross entropy ≈ 2.20 is equal to a perplexity ≈ 9 [ln(14)≈2.64, ln(13)≈2.56, ln(9)≈2.2]. This means that, the number of possible chords will drop from 14 possible choices (the number of different chords) to 9. This is a very reassuring discovery, because this result was obtained only by looking at the previous chord, and not offset, the bar number, multiple previous chords or knowing the start of the chorale. Adding enough dimensionality while creating the model could provide notably accurate results. That is possible, because practically in baroque harmony the next chord is based on the pattern of previous chords in various parts of a chorale. That logic can be adopted by using time series with enough and not too much depth (with more data than only the roman chords). For that reason a linear baseline and non-linear approaches such as LSTM and GRU models will be used. Judging from the cross-entropy results, the heat map, and the earlier work in the literature (almost none of the previous projects recommended plain RNNs), the choice of models is driven by the structure observed in the data rather than by trying architectures at random.
<br><br><br>
*Note, the cross_entropy/perplexity were computed by using the full corpus dataset, when modelling, the data will be split by chorale which means that the numbers displayed could be different from the random split set.*